# DQN Dino AI - Kaggle Training

Train Deep Q-Network cho Chrome Dino tren **Kaggle (GPU T4 x2, free)**.

---
**1. New Notebook > Accelerator > GPU T4 x2**
**2. Upload code: keo tha folder `shared/`, `dqn/`, `templates/` vao `/kaggle/working/` (panel Data ben phai, Add Data)**
**3. Hoac clone tu GitHub**
**4. Run all**
---

| Thoi gian | Episodes | Chat luong |
|-----------|----------|----------|
| ~20 phut  | 300      | Kha      |
| ~1 gio    | 800      | Tot      |
| ~2-3 gio  | 2000     | Rat tot  |

## 1. Cai dat dependencies

In [ ]:
!pip install pygame numpy torch matplotlib pillow -q
print('Done')

## 2. Upload project files

**Cach 1: Upload thu cong**
Keo tha 3 folder vao `/kaggle/working/` (kich thuoc ~200KB):
```
shared/       <- shared/config.py, shared/game_env.py, ...
dqn/          <- dqn/dqn_ai.py, dqn/train_dqn.py, ...
templates/    <- template_ai.py, ...
```

**Cach 2: Clone tu GitHub**

In [ ]:
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /kaggle/working/

import sys, os
sys.path.insert(0, '/kaggle/working')
print('Files trong /kaggle/working:', os.listdir('/kaggle/working'))

## 3. Kiem tra cau truc project

In [ ]:
required = [
    'shared/config.py',
    'shared/game_env.py',
    'shared/base_ai.py',
    'dqn/dqn_ai.py',
    'dqn/train_dqn.py',
    'template_ai.py',
]
all_ok = True
for p in required:
    ok = os.path.exists(f'/kaggle/working/{p}')
    status = 'OK' if ok else 'MISSING'
    print(f'  {status:7} /kaggle/working/{p}')
    if not ok:
        all_ok = False

if not all_ok:
    print('\nUpload cac folder missing vao Kaggle roi chay lai cell nay!')

## 4. Headless pygame setup

In [ ]:
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

import pygame
pygame.init()
pygame.display.set_mode((1, 1))  # fake window
print(f'pygame ready: SDL {pygame.get_sdl_version()}')

## 5. Xac nhan cau hinh

In [ ]:
from shared.config import STATE_SIZE, ACTION_SIZE, INIT_SPEED, MAX_SPEED, SPEED_INCREMENT

print(f'State: {STATE_SIZE} dims  |  Action: {ACTION_SIZE}')
print(f'Speed: {INIT_SPEED} -> {MAX_SPEED}  (inc {SPEED_INCREMENT}/step)')
print()

from dqn.dqn_ai import DQN_CONFIG

print('DQN_CONFIG:')
for k, v in DQN_CONFIG.items():
    print(f'  {k:22} = {v}')

## 6. Thu muc luu model

Tren Kaggle, model duoc luu vao `/kaggle/working/` — ban co the tai ve tu panel **Output** sau khi train xong.

In [ ]:
import os

MODEL_DIR = '/kaggle/working/dino_dqn_models'
MODEL_PATH = f'{MODEL_DIR}/dqn_best.pkl'
CHART_PATH = f'{MODEL_DIR}/training_curve.png'

os.makedirs(MODEL_DIR, exist_ok=True)

print(f'Model dir:  {MODEL_DIR}')
print(f'Model path: {MODEL_PATH}')
print(f'Chart path: {CHART_PATH}')

## 7. Khoi tao AI

Kiem tra GPU va khoi tao mang.

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

from dqn.dqn_ai import DQNDinoAI

ai = DQNDinoAI()
print(f'\nQ_net params:  {sum(p.numel() for p in ai.q_net.parameters()):,}')
print(f'Target_net:   {sum(p.numel() for p in ai.target_net.parameters()):,}')

## 8. Training

Doi `n_episodes` theo thoi gian ban co.
Model tot nhat tu dong duoc luu vao `/kaggle/working/dino_dqn_models/`.

In [ ]:
import time, numpy as np

start = time.time()

scores = ai.train(
    n_episodes       = 800,        # DOI SO EPISODES TAI DAY (khuyen nghi 800-2000)
    max_steps_per_ep = 10_000,
    verbose_every    = 50,
    save_path        = MODEL_PATH,
)

elapsed = time.time() - start
print(f'\nDONE in {elapsed/60:.1f} min')
print(f'Best score: {ai.best_score}')

## 9. Ve do thi tien do

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(scores, alpha=0.4, color='steelblue', label='Score moi episode')

window = 50
moving_avg = np.convolve(scores, np.ones(window) / window, mode='valid')
ax.plot(range(window - 1, len(scores)), moving_avg,
        color='orange', linewidth=2, label=f'Moving avg({window})')

ax.set_xlabel('Episode')
ax.set_ylabel('Score')
ax.set_title('DQN - Qua trinh hoc (Chrome Dino)')
ax.legend()
plt.tight_layout()
plt.savefig(CHART_PATH, dpi=150)
plt.show()
print(f'Da luu: {CHART_PATH}')

## 10. Danh gia

Chay 20 lan khong render, tinh thong ke.

In [ ]:
ai.epsilon = 0.0

from shared.evaluator import evaluate

stats = evaluate(ai, n_runs=20, verbose=True)
print(f'Tong ket: mean={stats["mean"]:.0f}  max={stats["max"]}  '
      f'min={stats["min"]}  std={stats["std"]:.0f}')

## 11. Test 1 episode

In [ ]:
from shared.game_env import DinoEnv, Dinosaur

env = DinoEnv(render=False)
dino = Dinosaur(env.sprites)
state = env.reset(dino)
done = False

while not done:
    action = ai.predict(state)
    state, _, done, _ = env.step_single(dino, action)

env.close()
print(f'Episode score: {dino.score}')

## 12. Tai model ve

Sau khi train xong, vao panel **Output** (ben phai) de tai cac file:
- `dino_dqn_models/dqn_best.pkl` — model tot nhat
- `dino_dqn_models/training_curve.png` — bieu do huan luyen

In [ ]:
!ls -lh /kaggle/working/dino_dqn_models/